# Tutorial 2: Using CauST on Your Own Data

This notebook shows how to apply CauST to **any** spatial transcriptomics dataset.

## Requirements
Your data must have:
1. A gene expression matrix (spots × genes)
2. 2D spatial coordinates per spot

Supported formats: `.h5ad`, `.h5` (10x), `.csv`, or any format readable by scanpy.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

from caust import CauST
from caust.data.loader import load_and_preprocess
from caust.evaluate.metrics import evaluate_single_slice

## Step 1: Load Your Data

CauST works with scanpy's AnnData format. Below are examples for common input formats.

In [ ]:
# ── Option A: Load from .h5ad file (most common) ──────────────────────
# adata = sc.read_h5ad("path/to/your_data.h5ad")

# ── Option B: Load 10x Visium .h5 + spatial/ folder ──────────────────
# adata = sc.read_visium("path/to/visium_folder/")

# ── Option C: Load from CSV (expression matrix + coordinates) ─────────
# expression_df = pd.read_csv("expression.csv", index_col=0)  # spots × genes
# coords_df = pd.read_csv("coordinates.csv", index_col=0)     # spots × (x, y)
# adata = ad.AnnData(X=expression_df.values.astype(np.float32))
# adata.obs_names = expression_df.index.tolist()
# adata.var_names = expression_df.columns.tolist()
# adata.obsm['spatial'] = coords_df[['x', 'y']].values.astype(np.float64)

# ── For this demo, we create synthetic data ──────────────────────────
from scipy.sparse import csr_matrix

rng = np.random.default_rng(42)
n_spots, n_genes, n_domains = 500, 300, 4
spots_per = n_spots // n_domains

X_parts = []
for i in range(n_domains):
    base = rng.poisson(lam=5, size=(spots_per, n_genes)).astype(np.float32)
    # Give each domain 25 unique marker genes
    marker_start = i * 25
    base[:, marker_start:marker_start + 25] += 8 * (i + 1)
    X_parts.append(base)

X = np.vstack(X_parts)
adata = ad.AnnData(X=csr_matrix(X))
adata.obs_names = [f'spot_{i}' for i in range(n_spots)]
adata.var_names = [f'gene_{j}' for j in range(n_genes)]

# Spatial coordinates: 4 spatially separated clusters
coords = np.vstack([
    rng.normal([0, 0], 2, (spots_per, 2)),
    rng.normal([20, 0], 2, (spots_per, 2)),
    rng.normal([0, 20], 2, (spots_per, 2)),
    rng.normal([20, 20], 2, (spots_per, 2)),
])
adata.obsm['spatial'] = coords

# Optional: ground-truth labels (for evaluation only)
adata.obs['ground_truth'] = np.repeat(range(n_domains), spots_per).astype(str)

print(f'Loaded: {adata.n_obs} spots × {adata.n_vars} genes')
print(f'Spatial shape: {adata.obsm["spatial"].shape}')
adata

## Step 2: Preprocess

CauST requires standard scanpy preprocessing. You can either:
- Use CauST's built-in `load_and_preprocess()`, or
- Do it yourself with scanpy

In [ ]:
# ── Option A: CauST's built-in preprocessor ──────────────────────────
adata_pp = load_and_preprocess(
    adata,
    n_top_genes = 200,    # number of highly variable genes to keep
    min_genes   = 10,     # min genes per spot
    min_cells   = 3,      # min spots per gene
    normalize   = True,
    scale       = True,
)

# ── Option B: Manual scanpy preprocessing ────────────────────────────
# sc.pp.filter_cells(adata, min_genes=200)
# sc.pp.filter_genes(adata, min_cells=3)
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, n_top_genes=3000)
# adata = adata[:, adata.var.highly_variable]
# sc.pp.scale(adata)

print(f'After preprocessing: {adata_pp.n_obs} spots × {adata_pp.n_vars} genes')

## Step 3: Choose Your Hyperparameters

| Parameter | Description | Recommended |
|-----------|-------------|-------------|
| `n_causal_genes` | Number of top causal genes to keep | 200–500 |
| `n_clusters` | Expected number of spatial domains | Based on your tissue |
| `scoring_method` | `'perturbation'` (accurate) or `'gradient'` (fast) | `'perturbation'` for publication, `'gradient'` for exploration |
| `filter_mode` | `'filter'`, `'reweight'`, or `'filter_and_reweight'` | `'filter_and_reweight'` |
| `epochs` | Training epochs for GAT autoencoder | 300–500 |
| `alpha` | Weight for invariance vs causal score (multi-slice only) | 0.5 |

In [ ]:
model = CauST(
    n_causal_genes  = 50,           # adjust for your dataset
    n_clusters      = n_domains,    # number of expected domains
    epochs          = 100,          # increase for real data
    scoring_method  = 'gradient',   # 'perturbation' for best results
    filter_mode     = 'filter_and_reweight',
    verbose         = True,
)

## Step 4: Run CauST

In [ ]:
adata_out = model.fit_transform(adata_pp.copy())

print(f'\nOutput: {adata_out.n_obs} spots × {adata_out.n_vars} genes')
print(f'Domain labels: {np.unique(adata_out.obs["caust_domain"].values)}')
print(f'Latent embedding: {adata_out.obsm["caust_latent"].shape}')

## Step 5: Inspect Causal Genes

CauST identifies which genes causally drive domain formation rather than being merely correlated.

In [ ]:
# Top 20 causal genes
top_genes = model.get_top_causal_genes(n=20)
print('Top 20 causal genes by score:')
for gene, score in top_genes:
    print(f'  {gene}: {score:.4f}')

# Full scores dict
all_scores = model.get_causal_scores()
print(f'\nTotal genes scored: {len(all_scores)}')

## Step 6: Evaluate (if ground truth available)

In [ ]:
labels_pred = adata_out.obs['caust_domain'].astype(int).values
latent_Z    = adata_out.obsm['caust_latent']

# With ground truth
if 'ground_truth' in adata_pp.obs.columns:
    labels_true = adata_pp.obs.loc[adata_out.obs_names, 'ground_truth'].values
    metrics = evaluate_single_slice(labels_pred, latent_Z, labels_true)
else:
    # Without ground truth — silhouette only
    metrics = evaluate_single_slice(labels_pred, latent_Z, labels_true=None)

print('Evaluation metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## Step 7: Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
coords = adata_pp.obsm['spatial']

# 1. Ground truth (if available)
ax = axes[0]
if 'ground_truth' in adata_pp.obs.columns:
    gt = adata_pp.obs['ground_truth'].astype(int).values
    ax.scatter(coords[:, 0], coords[:, 1], c=gt, cmap='tab10', s=8, alpha=0.8)
    ax.set_title('Ground Truth Domains')
else:
    ax.text(0.5, 0.5, 'No ground truth', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Ground Truth (N/A)')
ax.set_aspect('equal')

# 2. CauST predicted domains
ax = axes[1]
pred = adata_out.obs['caust_domain'].astype(int).values
ax.scatter(coords[:len(pred), 0], coords[:len(pred), 1], c=pred, cmap='tab10', s=8, alpha=0.8)
ax.set_title('CauST Predicted Domains')
ax.set_aspect('equal')

# 3. Top causal gene scores
ax = axes[2]
genes, scores = zip(*model.get_top_causal_genes(n=15))
ax.barh(range(len(genes)), scores, color='steelblue')
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes, fontsize=8)
ax.set_xlabel('Causal Score')
ax.set_title('Top 15 Causal Genes')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Step 8: Compare Filter Modes

CauST supports three gene selection strategies. Compare them to find the best for your data.

In [ ]:
modes = ['filter', 'reweight', 'filter_and_reweight']
comparison = []

for mode in modes:
    m = CauST(
        n_causal_genes=50, n_clusters=n_domains, epochs=100,
        scoring_method='gradient', filter_mode=mode, verbose=False,
    )
    out = m.fit_transform(adata_pp.copy())
    pred = out.obs['caust_domain'].astype(int).values
    Z = out.obsm['caust_latent']
    
    gt = adata_pp.obs.loc[out.obs_names, 'ground_truth'].values if 'ground_truth' in adata_pp.obs else None
    met = evaluate_single_slice(pred, Z, gt)
    met['mode'] = mode
    comparison.append(met)
    print(f'{mode:25s}  ARI={met.get("ari", 0):.4f}  Sil={met.get("silhouette", 0):.4f}')

comp_df = pd.DataFrame(comparison)
comp_df

## Step 9: Save & Reload Your Model

In [ ]:
# Save
model.save('my_caust_model')
print('Model saved to my_caust_model/')

# Reload (n_genes auto-detected from config)
loaded = CauST.load('my_caust_model')

# Apply to new data
new_out = loaded.transform(adata_pp.copy())
print(f'Reloaded model: {new_out.n_obs} spots, {new_out.n_vars} genes')
print('Labels match:', np.array_equal(
    new_out.obs['caust_domain'].values,
    adata_out.obs['caust_domain'].values
))

In [ ]:
# Clean up saved model (optional)
import shutil
shutil.rmtree('my_caust_model', ignore_errors=True)

## Step 10: Export Results

Save outputs for downstream analysis or publication.

In [ ]:
# Save domain labels as CSV
adata_out.obs[['caust_domain']].to_csv('domain_labels.csv')

# Save causal gene scores as CSV
scores_df = pd.DataFrame(
    model.get_top_causal_genes(n=len(all_scores)),
    columns=['gene', 'causal_score']
)
scores_df.to_csv('causal_gene_scores.csv', index=False)

# Save processed AnnData (includes latent embedding + domain labels)
# adata_out.write_h5ad('results.h5ad')

print('Results exported!')
print(f'  domain_labels.csv: {len(adata_out)} spots')
print(f'  causal_gene_scores.csv: {len(scores_df)} genes')

In [ ]:
# Clean up exported files (optional)
import os
for f in ['domain_labels.csv', 'causal_gene_scores.csv']:
    if os.path.exists(f):
        os.remove(f)

## Checklist: Bringing Your Own Data

- [ ] Data in `.h5ad` or readable by scanpy (rows = spots, columns = genes)
- [ ] 2D spatial coordinates stored in `adata.obsm['spatial']`
- [ ] Run preprocessing: normalize, log1p, HVG selection, scale
- [ ] Choose `n_clusters` based on expected tissue anatomy
- [ ] Start with `scoring_method='gradient'` for speed; switch to `'perturbation'` for publication
- [ ] If you have multiple slices, use `model.fit_multi_slice()` for invariance scoring
- [ ] If ground-truth labels exist, add them to `adata.obs` for ARI/NMI evaluation

## Next Steps

- Multi-slice mode: see `scripts/04_run_multi_slice.py`
- Full benchmark: see `scripts/05_benchmark.py`
- Integration with STAGATE/GraphST: see `caust/models/stagate_wrapper.py`